In [5]:
import torch
import platform
import os
from pathlib import Path
from safetensors.torch import save_model
import json
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

In [6]:
def get_device_details():
    """This is For Get Device Details"""
    print("=" * 60)
    print("PyTorch Device Information")
    print("=" * 60)

    print(f"PyTorch version : {torch.__version__}")

    if torch.cuda.is_available():
        print("Device          : CUDA")
        print(f"GPU count       : {torch.cuda.device_count()}")

        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)

            print(f"\nGPU {i}")
            print(f"  Name          : {props.name}")
            print(f"  Device        : cuda:{i}")
            print(f"  Compute       : {props.major}.{props.minor}")
            print(f"  VRAM          : {props.total_memory / (1024**3):.2f} GB")
            print(f"  Multiprocessor : {props.multi_processor_count}")

        print(f"\nCUDA version    : {torch.version.cuda}")
        print(f"cuDNN version   : {torch.backends.cudnn.version()}")

        # Current GPU memory usage
        print("\nCurrent GPU Memory")
        print(f"  Allocated      : "
              f"{torch.cuda.memory_allocated() / (1024**2):.2f} MB")
        print(f"  Reserved       : "
              f"{torch.cuda.memory_reserved() / (1024**2):.2f} MB")

    else:
        print("Device          : CPU")
        print(f"CPU             : {platform.processor()}")
        print(f"Architecture    : {platform.machine()}")
        print(f"CPU cores       : {os.cpu_count()}")
        print("CUDA available  : False")

    print("=" * 60)

def pt_to_safetensors(model ,weights_dir : str):
    for filename in os.listdir(weights_dir):
        if filename.endswith(".pt"):
            ckpt = torch.load(weights_dir + "\\" + filename)
            if isinstance(ckpt,dict) and "model_state_dict" in ckpt:
                state_dict = ckpt["model_state_dict"]
                model.load_state_dict(state_dict)
                save_model(model, weights_dir + "\\"+ filename.removesuffix(".pt") +  ".safetensors")

def save_model_outputs(
    prompts: list[str],
    generated_outputs: list[str],
    filepath: str,
    model_name: str = "Peak-7M",
    extra_info: dict | None = None,
):
    """
    Save prompts and model-generated outputs for evaluation.

    Args:
        prompts: List of input prompts.
        generated_outputs: List of model-generated texts.
        filepath: Output JSON file.
        model_name: Name/version of your model.
        extra_info: Optional training/model information.
    """

    if len(prompts) != len(generated_outputs):
        raise ValueError(
            f"Number of prompts ({len(prompts)}) must equal "
            f"number of outputs ({len(generated_outputs)})"
        )

    samples = []

    for i, (prompt, output) in enumerate(
        zip(prompts, generated_outputs), start=1
    ):
        samples.append({
            "id": i,
            "prompt": prompt,
            "generated_text": output,
        })

    evaluation_data = {
        "model": {
            "name": model_name,
            "evaluation_date": datetime.now().isoformat(),
            "extra_info": extra_info or {},
        },
        "num_samples": len(samples),
        "samples": samples,
    }

    filepath = Path(filepath)

    with filepath.open("w", encoding="utf-8") as f:
        json.dump(
            evaluation_data,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"Saved {len(samples)} samples to: {filepath}")

In [9]:
# Config.from_json(path="exp_1/exp_1_config.json")

PeakConfig(model_type='Decoder Transformer', model_name='Peak', vocab_size=4096, ctx_length=512, d_model=256, n_layers=6, n_heads=8, n_kv_heads=1, d_ff=2048)

In [ ]:
# # Checkpointing
#     save_every_steps: int
#     validate_every_steps: int
#     checkpoint_dir: str
#
#     # LR scheduler
#     total_steps: int
#     warmup_steps: int
#     min_learning_rate: float